In [1]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch

from open_clip import create_model_and_transforms, get_tokenizer


In [2]:
device = "cuda"

model_name = "coca_ViT-L-14"
pretrained_path = "/project_antwerp/hbae/open_clip2/logs/finetune_HEG_final/checkpoints/epoch_5.pt"

model, preprocess_train, preprocess_val = create_model_and_transforms(
    model_name,
    pretrained=pretrained_path
)
tokenizer = get_tokenizer(model_name)
model = model.to(device)
model.eval()

print("Model loaded:", model_name)


Model loaded: coca_ViT-L-14


In [3]:
train_df = pd.read_csv("/project_antwerp/hbae/data/HEG_finetune_meta_gpulab.csv")
print(train_df.shape)
train_df.head()


(154763, 6)


,Unnamed: 0,label,sample_name,dataset,img_idx,img_path
0,GSM5494475_AAACACCAATAACTGC-1_hires,TTN IGKC MT-ND4 MT-CO1 MT-CO3 ACTA1 MT-ATP6 NE...,GSM5494475,GSE181300,GSM5494475_AAACACCAATAACTGC-1_hires,/project_antwerp/hbae/data/Processed_Data/GSE1...
1,GSM5494475_AAACAGCTTTCAGAAG-1_hires,IGKC IGHG4 IGLC2 IGHG1 IGHG3 COL1A2 COL1A1 THB...,GSM5494475,GSE181300,GSM5494475_AAACAGCTTTCAGAAG-1_hires,/project_antwerp/hbae/data/Processed_Data/GSE1...
2,GSM5494475_AAACAGGGTCTATATT-1_hires,IGKC IGHG4 IGHG3 IGLC2 B2M ACTB MALAT1 AQP1 IG...,GSM5494475,GSE181300,GSM5494475_AAACAGGGTCTATATT-1_hires,/project_antwerp/hbae/data/Processed_Data/GSE1...
3,GSM5494475_AAACATGGTGAGAGGA-1_hires,IGKC IGHG4 IGHG3 IGLC2 MALAT1 MT-CO3 TNNT3 MT-...,GSM5494475,GSE181300,GSM5494475_AAACATGGTGAGAGGA-1_hires,/project_antwerp/hbae/data/Processed_Data/GSE1...
4,GSM5494475_AAACCGGGTAGGTACC-1_hires,IGKC IGHG4 IGHG3 IGLC2 IGHG1 CD74 FTL IGLC1 MA...,GSM5494475,GSE181300,GSM5494475_AAACCGGGTAGGTACC-1_hires,/project_antwerp/hbae/data/Processed_Data/GSE1...


In [4]:
def get_text_emb(sentences):
    all_emb = []

    for text in tqdm(sentences):
        tokens = tokenizer([text]).to(device)
        with torch.no_grad():
            emb = model.encode_text(tokens)
            emb = emb / emb.norm(dim=-1, keepdim=True)
        all_emb.append(emb.cpu().numpy()[0])
    
    return np.vstack(all_emb)

train_sentences = train_df["label"].astype(str).tolist()
text_emb = get_text_emb(train_sentences)
print(text_emb.shape)

np.save("train_text_embedding.npy", text_emb)


100%|██████████| 154763/154763 [14:14<00:00, 181.20it/s]


(154763, 768)


In [11]:

from pathlib import Path
import h5py
import numpy as np
import torch
from tqdm import tqdm
from PIL import Image

def load_hdf5_patches(hdf5_path):
    """
    HDF5 파일 안에서 '이미지처럼 생긴(차원 3 or 4)' dataset 하나를 자동으로 찾아서 반환.
    """
    f = h5py.File(hdf5_path, "r")
    candidates = []

    def collect(name, obj):
        # dataset 이고, 차원이 3( H, W, C ) 또는 4( N, H, W, C / N, C, H, W ) 이상인 애들만 후보로
        if isinstance(obj, h5py.Dataset) and obj.ndim >= 3:
            candidates.append(name)

    f.visititems(collect)

    if not candidates:
        f.close()
        raise KeyError(f"No image-like dataset (ndim>=3) found in HDF5 file: {hdf5_path}")

    # 일단 첫 번째 후보를 사용 (필요하면 출력해서 확인 가능)
    key = candidates[0]
    print(f"▶ Using dataset '{key}' from {hdf5_path}")
    imgs = f[key]     # shape: (N, H, W, C) or (N, C, H, W) or 비슷한 형태

    return f, imgs

from pathlib import Path

# 슬라이드 patch 폴더들이 모여 있는 루트
tcga_root = Path("/project_antwerp/TCGA-HNSC/TCGA_patch/")

# 각 슬라이드 폴더 안의 모든 .h5 / .hdf5 파일 수집
h5_list = sorted(list(tcga_root.rglob("*.hdf5")))

print("HDF5 파일 개수:", len(h5_list))
for p in h5_list[:5]:
    print(p)






HDF5 파일 개수: 528
/project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-4P-AA8J-01Z-00-DX1/TCGA-4P-AA8J-01Z-00-DX1.hdf5
/project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-BA-4074-01Z-00-DX1/TCGA-BA-4074-01Z-00-DX1.hdf5
/project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-BA-4074-01Z-00-DX1.377dc7cb-f029-4bf5-bdd5-8997f91e551a/TCGA-BA-4074-01Z-00-DX1.hdf5
/project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-BA-4077-01Z-00-DX1/TCGA-BA-4077-01Z-00-DX1.hdf5
/project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-BA-4077-01Z-00-DX1.604893b7-5fcb-4bab-b2cd-6cad1c8ab456/TCGA-BA-4077-01Z-00-DX1.hdf5


In [12]:
from pathlib import Path

tcga_root = Path("/project_antwerp/TCGA-HNSC/TCGA_patch")

h5_list = sorted(list(tcga_root.rglob("*.h5")) + list(tcga_root.rglob("*.hdf5")))
print("총 HDF5:", len(h5_list))

# slide_id = 파일 이름(확장자 제거)
slide_to_path = {}

for p in h5_list:
    slide_id = p.stem  # 'TCGA-BA-4074-01Z-00-DX1'
    cur = slide_to_path.get(slide_id)

    if cur is None:
        slide_to_path[slide_id] = p
    else:
        # 보통 UUID 안 붙은 “짧은 경로”를 우선으로 사용
        if len(str(p)) < len(str(cur)):
            slide_to_path[slide_id] = p

print("슬라이드당 하나로 dedup 후 개수:", len(slide_to_path))
dedup_h5_list = list(slide_to_path.values())[:5]
for p in dedup_h5_list:
    print(p)


총 HDF5: 528
슬라이드당 하나로 dedup 후 개수: 331
/project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-4P-AA8J-01Z-00-DX1/TCGA-4P-AA8J-01Z-00-DX1.hdf5
/project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-BA-4074-01Z-00-DX1/TCGA-BA-4074-01Z-00-DX1.hdf5
/project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-BA-4077-01Z-00-DX1/TCGA-BA-4077-01Z-00-DX1.hdf5
/project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-BA-5151-01Z-00-DX1/TCGA-BA-5151-01Z-00-DX1.hdf5
/project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-BA-5153-01Z-00-DX1/TCGA-BA-5153-01Z-00-DX1.hdf5


In [7]:
import pandas as pd
import numpy as np

tcga_rna_csv = "/project_antwerp/TCGA-HNSC/ref_file.csv"  # 너가 실제 쓰는 경로로
rna_df = pd.read_csv(tcga_rna_csv)

# slide_id 정리
if 'wsi_file_name' in rna_df.columns:
    rna_df['wsi_file_name'] = rna_df['wsi_file_name'].astype(str)
    rna_df['wsi_file_name'] = rna_df['wsi_file_name'].str.split('/').str[-1]
    rna_df['slide_id'] = rna_df['wsi_file_name'].str.split('.').str[0]
    rna_df = rna_df.set_index('slide_id')
elif 'slide_id' in rna_df.columns:
    rna_df = rna_df.set_index('slide_id')
else:
    first_col = rna_df.columns[0]
    rna_df = rna_df.set_index(first_col)

# 숫자 컬럼만 남기고 정리 (전에 하던 대로)
if 'Unnamed: 0' in rna_df.columns:
    rna_df = rna_df.drop(columns=['Unnamed: 0'])
if 'patient_id' in rna_df.columns:
    rna_df = rna_df.drop(columns=['patient_id'])

rna_df = rna_df.select_dtypes(include=[np.number])

print("RNA shape:", rna_df.shape)
print("슬라이드 수:", len(rna_df))


RNA shape: (461, 24778)
슬라이드 수: 461


In [8]:
import pandas as pd

# 예: rna_df.index 가 slide_id 들 (461개)
rna_slide_ids = set(rna_df.index.astype(str))

# dedup된 HDF5 중에서 RNA가 있는 슬라이드만 선택
matched_h5_paths = [p for sid, p in slide_to_path.items() if sid in rna_slide_ids]

print("RNA와 매칭된 슬라이드 수:", len(matched_h5_paths))
for p in matched_h5_paths[:5]:
    print(p)


RNA와 매칭된 슬라이드 수: 331
/project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-4P-AA8J-01Z-00-DX1/TCGA-4P-AA8J-01Z-00-DX1.hdf5
/project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-BA-4074-01Z-00-DX1/TCGA-BA-4074-01Z-00-DX1.hdf5
/project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-BA-4077-01Z-00-DX1/TCGA-BA-4077-01Z-00-DX1.hdf5
/project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-BA-5151-01Z-00-DX1/TCGA-BA-5151-01Z-00-DX1.hdf5
/project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-BA-5153-01Z-00-DX1/TCGA-BA-5153-01Z-00-DX1.hdf5


In [15]:
import h5py
import numpy as np

class H5ImageList:
    """여러 3D dataset(각각이 한 patch)을 list처럼 다루기 위한 래퍼"""
    def __init__(self, h5file, keys):
        self.h5file = h5file
        self.keys = keys  # 패치 이름 리스트

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, idx):
        # h5py Dataset -> numpy array로 로드
        return self.h5file[self.keys[idx]][:]

def load_hdf5_patches(hdf5_path):
    f = h5py.File(hdf5_path, "r")

    # 1) 먼저 4D batched dataset이 있는지 확인 (N, H, W, C) or (N, C, H, W)
    for k in f.keys():
        ds = f[k]
        if isinstance(ds, h5py.Dataset) and ds.ndim == 4:
            # 기존 코드와 동일하게 동작하도록 그대로 반환
            return f, ds

    # 2) 아니면, 여러 개의 3D (H, W, C) patch dataset으로 이루어진 케이스 처리
    img_keys = [
        k for k in f.keys()
        if isinstance(f[k], h5py.Dataset) and f[k].ndim == 3
    ]

    if img_keys:
        # 순서 고정하고 싶으면 정렬
        img_keys = sorted(img_keys)  # 이름으로 정렬 (예: "89600_50176", ...)
        imgs = H5ImageList(f, img_keys)
        return f, imgs

    # 3) 아무것도 못 찾으면 에러
    f.close()
    raise KeyError(f"No image dataset found in HDF5 file: {hdf5_path}")


In [16]:
from PIL import Image
from tqdm import tqdm
import torch
import numpy as np  # ← 이거!

def extract_embeddings_from_hdf5(model, preprocess_val, hdf5_path, device):
    f, imgs = load_hdf5_patches(hdf5_path)
    emb_list = []

    print(f"Found {len(imgs)} patches in {hdf5_path}")

    for i in tqdm(range(len(imgs))):
        arr = imgs[i]  # numpy array (H, W, C) or (C, H, W) 또는 래퍼에서 가져온 것

        # If array is CHW, convert to HWC
        if arr.ndim == 3 and arr.shape[0] == 3:
            arr = np.transpose(arr, (1, 2, 0))

        img = Image.fromarray(arr.astype("uint8"))
        img = preprocess_val(img).unsqueeze(0).to(device)

        with torch.no_grad():
            emb = model.encode_image(img)
            emb = emb / emb.norm(dim=1, keepdim=True)

        emb_list.append(emb.cpu().numpy()[0])

    f.close()
    return np.vstack(emb_list)
def extract_embeddings_from_hdf5_batched(model, preprocess_val, hdf5_path, device, batch_size=64):
    f, imgs = load_hdf5_patches(hdf5_path)
    print(f"Found {len(imgs)} patches in {hdf5_path}")

    emb_out = []
    batch_tensors = []

    for i in tqdm(range(len(imgs))):
        arr = imgs[i]
        if arr.ndim == 3 and arr.shape[0] == 3:
            arr = np.transpose(arr, (1, 2, 0))

        img = Image.fromarray(arr.astype("uint8"))
        batch_tensors.append(preprocess_val(img))

        if len(batch_tensors) == batch_size or i == len(imgs) - 1:
            x = torch.stack(batch_tensors, dim=0).to(device)
            with torch.no_grad():
                emb = model.encode_image(x)
                emb = emb / emb.norm(dim=1, keepdim=True)
            emb_out.append(emb.cpu().numpy())
            batch_tensors = []

    f.close()
    return np.vstack(emb_out)



In [26]:
import os
import numpy as np
import h5py
from PIL import Image
from tqdm import tqdm
import torch
from pathlib import Path
from contextlib import nullcontext


def load_hdf5_patches(hdf5_path):
    """
    Returns:
      f: h5py.File
      imgs: either
        - h5py.Dataset (shape: (N,H,W,C) or (N,C,H,W))
        - list[str] keys where each key is a patch dataset (shape: (H,W,C))
    """
    f = h5py.File(hdf5_path, "r")

    # Case A: common single-dataset layout
    candidate_keys = ["imgs", "images", "img", "patches", "data", "x"]
    for k in candidate_keys:
        if k in f and isinstance(f[k], h5py.Dataset):
            return f, f[k]

    # Case B: root has many datasets per patch (your case: "89600_50176", ...)
    keys = []
    for k in f.keys():
        obj = f[k]
        if isinstance(obj, h5py.Dataset):
            shp = obj.shape
            # typically (256,256,3) or (H,W,3)
            if len(shp) == 3 and shp[-1] in (1, 3, 4):
                keys.append(k)

    if len(keys) == 0:
        f.close()
        raise KeyError(f"No image dataset found in HDF5 file: {hdf5_path}")

    keys.sort()
    return f, keys


def extract_embeddings_from_hdf5(
    model,
    preprocess_val,
    hdf5_path,
    device,
    batch_size=64,
    use_fp16=True,
    verbose_skip=False,
):
    device = torch.device(device) if isinstance(device, str) else device
    f, imgs = load_hdf5_patches(hdf5_path)

    is_dataset = isinstance(imgs, h5py.Dataset)
    n = imgs.shape[0] if is_dataset else len(imgs)

    print(f"Found {n} patches in {hdf5_path}")

    # FP16 only on CUDA
    autocast_enabled = (use_fp16 and device.type == "cuda")
    autocast_ctx = torch.cuda.amp.autocast if autocast_enabled else nullcontext

    def _get_patch(i):
        if is_dataset:
            return imgs[i]          # returns numpy-like array
        else:
            return f[imgs[i]][()]   # dataset -> numpy array

    emb_chunks = []
    batch_tensors = []
    skipped = 0

    model.eval()

    for i in tqdm(range(n)):
        try:
            arr = _get_patch(i)

            # if CHW -> HWC
            if arr.ndim == 3 and arr.shape[0] in (1, 3, 4) and arr.shape[-1] not in (1, 3, 4):
                arr = np.transpose(arr, (1, 2, 0))

            # ensure uint8
            if arr.dtype != np.uint8:
                arr = arr.astype(np.uint8)

            img = Image.fromarray(arr)
            batch_tensors.append(preprocess_val(img))

        except Exception as e:
            skipped += 1
            if verbose_skip:
                print(f"[skip] patch {i}: {e}")
            continue

        # flush batch
        if len(batch_tensors) == batch_size or i == n - 1:
            x = torch.stack(batch_tensors, dim=0).to(device, non_blocking=True)
            batch_tensors = []

            with torch.no_grad():
                with autocast_ctx():
                    emb = model.encode_image(x)

                emb = emb / emb.norm(dim=1, keepdim=True)
                emb_chunks.append(emb.float().cpu().numpy())

    f.close()

    if skipped > 0:
        print(f"⚠️ Skipped patches due to errors: {skipped}/{n}")

    if len(emb_chunks) == 0:
        # return empty (safer than crash)
        return np.zeros((0, 0), dtype=np.float32)

    return np.vstack(emb_chunks)


def atomic_save_npy(out_path: Path, arr: np.ndarray):
    """
    Safe save: write to temp then atomic replace.
    Fixes: np.save auto-appending .npy if given a string path.
    """
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    tmp_path = out_path.with_suffix(out_path.suffix + ".tmp")  # e.g. file.npy.tmp

    # IMPORTANT: use file handle so np.save doesn't append ".npy"
    with open(tmp_path, "wb") as ftmp:
        np.save(ftmp, arr)

    os.replace(tmp_path, out_path)  # atomic on POSIX


# =========================
# Main loop (resume-capable)
# =========================

save_dir = Path("./tcga_embeddings")
save_dir.mkdir(exist_ok=True)

all_embeddings = {}

for h5 in matched_h5_paths:
    h5 = Path(h5)
    slide_id = h5.stem.split(".")[0]
    out_path = save_dir / f"{slide_id}.npy"

    # resume: already exists -> skip
    if out_path.exists():
        print(f"⏭️ Skip (exists): {slide_id}")
        try:
            all_embeddings[slide_id] = np.load(out_path, mmap_mode="r")
        except Exception:
            pass
        continue

    print(f"\nProcessing slide: {slide_id}")

    try:
        emb = extract_embeddings_from_hdf5(
            model=model,
            preprocess_val=preprocess_val,
            hdf5_path=h5,
            device=device,
            batch_size=64,     # tune: 32/64/128
            use_fp16=True,     # fp16 only if cuda
            verbose_skip=False
        )

        atomic_save_npy(out_path, emb)
        all_embeddings[slide_id] = emb
        print(f"✅ Saved: {out_path} | shape={emb.shape}")

    except KeyError as e:
        print(f"❌ Skipping (no image dataset): {h5} | {e}")
        continue
    except Exception as e:
        print(f"❌ Error on {slide_id}: {e}")
        continue

print("✅ 전체 슬라이드 embedding 완료")


⏭️ Skip (exists): TCGA-4P-AA8J-01Z-00-DX1
⏭️ Skip (exists): TCGA-BA-4074-01Z-00-DX1
⏭️ Skip (exists): TCGA-BA-4077-01Z-00-DX1
⏭️ Skip (exists): TCGA-BA-5151-01Z-00-DX1
⏭️ Skip (exists): TCGA-BA-5153-01Z-00-DX1
⏭️ Skip (exists): TCGA-BA-6870-01Z-00-DX1

Processing slide: TCGA-BA-7269-01Z-00-DX1
Found 6126 patches in /project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-BA-7269-01Z-00-DX1/TCGA-BA-7269-01Z-00-DX1.hdf5


100%|██████████| 6126/6126 [00:28<00:00, 216.37it/s]


✅ Saved: tcga_embeddings/TCGA-BA-7269-01Z-00-DX1.npy | shape=(6126, 768)

Processing slide: TCGA-BA-A6D8-01Z-00-DX1
Found 11278 patches in /project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-BA-A6D8-01Z-00-DX1/TCGA-BA-A6D8-01Z-00-DX1.hdf5


100%|██████████| 11278/11278 [01:50<00:00, 102.30it/s]


✅ Saved: tcga_embeddings/TCGA-BA-A6D8-01Z-00-DX1.npy | shape=(11278, 768)

Processing slide: TCGA-BA-A6DA-01Z-00-DX1
Found 9139 patches in /project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-BA-A6DA-01Z-00-DX1/TCGA-BA-A6DA-01Z-00-DX1.hdf5


...

✅ Saved: tcga_embeddings/TCGA-WA-A7GZ-01Z-00-DX2.npy | shape=(2164, 768)

Processing slide: TCGA-WA-A7H4-01Z-00-DX1
Found 7152 patches in /project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-WA-A7H4-01Z-00-DX1/TCGA-WA-A7H4-01Z-00-DX1.hdf5


100%|██████████| 7152/7152 [00:38<00:00, 184.81it/s]


✅ Saved: tcga_embeddings/TCGA-WA-A7H4-01Z-00-DX1.npy | shape=(7152, 768)

Processing slide: TCGA-WA-A7H4-01Z-00-DX2
Found 1948 patches in /project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-WA-A7H4-01Z-00-DX2/TCGA-WA-A7H4-01Z-00-DX2.hdf5


100%|██████████| 1948/1948 [00:10<00:00, 182.72it/s]

✅ Saved: tcga_embeddings/TCGA-WA-A7H4-01Z-00-DX2.npy | shape=(1948, 768)
✅ 전체 슬라이드 embedding 완료


In [2]:
import numpy as np
from pathlib import Path
import sys

# Loki 모듈 import
loki_path = "/project_antwerp/hbae/Loki/src"
if loki_path not in sys.path:
    sys.path.insert(0, loki_path)

try:
    import loki.predex
    print("✅ Loki 모듈 로드 성공")
except ImportError:
    print("⚠️ Loki 모듈을 찾을 수 없습니다. 직접 구현합니다.")
    def predict_st_gene_expr(similarity_matrix, train_data):
        """Loki PredEx 함수 직접 구현"""
        similarity_matrix = np.asarray(similarity_matrix)
        train_data = np.asarray(train_data)
        weighted_sum = similarity_matrix @ train_data
        weights = similarity_matrix.sum(axis=1, keepdims=True) + 1e-8
        return weighted_sum / weights
    loki = type('obj', (object,), {'predex': type('obj', (object,), {'predict_st_gene_expr': predict_st_gene_expr})()})

# =========================
# Config
# =========================
EMB_DIR = Path("tcga_embeddings")
OUT_DIR = Path("tcga_predicted_expression_loki")  # Loki 방식 결과 저장
OUT_DIR.mkdir(exist_ok=True)

EPS = 1e-8
PATCH_CHUNK = 256    # 메모리 효율을 위한 청크 크기

# =========================
# Load train-side data
# =========================
train_text_emb = np.load("train_text_embedding.npy").astype(np.float32)  # (Ntrain, 768)
train_expr     = np.load("/project_antwerp/hbae/data/combined_expression_matrix.npy").astype(np.float32)  # (Ntrain, G)

print("Train text:", train_text_emb.shape)
print("Train expr:", train_expr.shape)

# L2 normalize (cosine similarity = dot product)
train_text_emb /= (np.linalg.norm(train_text_emb, axis=1, keepdims=True) + EPS)

Ntrain, D = train_text_emb.shape
G = train_expr.shape[1]

print(f"✅ 데이터 로드 완료: {Ntrain}개 학습 샘플, {G}개 유전자")

# =========================
# Slide loop - Loki 방식
# =========================
for emb_file in sorted(EMB_DIR.glob("*.npy")):
    if emb_file.name.endswith(".npy.tmp"):
        continue

    slide_id = emb_file.stem
    out_path = OUT_DIR / f"{slide_id}.npy"

    # resume
    if out_path.exists():
        print(f"⏭️ Skip (exists): {slide_id}")
        continue

    print(f"\n🔹 Processing slide: {slide_id}")

    # 이미지 임베딩 로드 및 정규화
    img_emb = np.load(emb_file).astype(np.float32)  # (num_patches, 768)
    img_emb /= (np.linalg.norm(img_emb, axis=1, keepdims=True) + EPS)

    P = img_emb.shape[0]
    pred_chunks = []

    # 패치별로 청크 단위 처리 (메모리 효율)
    for p0 in range(0, P, PATCH_CHUNK):
        p1 = min(p0 + PATCH_CHUNK, P)
        img_chunk = img_emb[p0:p1]  # (chunk_size, 768)
        
        # 🔑 Loki 방식: 모든 similarity 계산 (Top-K 없음)
        similarity = img_chunk @ train_text_emb.T  # (chunk_size, Ntrain)
        
        # 🔑 Loki PredEx 함수 사용
        pred_chunk = loki.predex.predict_st_gene_expr(similarity, train_expr)  # (chunk_size, G)
        pred_chunks.append(pred_chunk)

    pred_expr = np.vstack(pred_chunks)  # (P, G)
    
    # 저장
    tmp_path = OUT_DIR / f"{slide_id}.tmp.npy"
    np.save(tmp_path, pred_expr)
    tmp_path.replace(out_path)

    print(f"✅ Saved predicted expression: {pred_expr.shape}")

print("\n✅ Loki 방식으로 전체 슬라이드 예측 완료")


✅ Loki 모듈 로드 성공
Train text: (154763, 768)
Train expr: (154763, 12009)
✅ 데이터 로드 완료: 154763개 학습 샘플, 12009개 유전자

🔹 Processing slide: TCGA-4P-AA8J-01Z-00-DX1
✅ Saved predicted expression: (11834, 12009)

🔹 Processing slide: TCGA-BA-4074-01Z-00-DX1
✅ Saved predicted expression: (1033, 12009)

🔹 Processing slide: TCGA-BA-4077-01Z-00-DX1
✅ Saved predicted expression: (1749, 12009)

🔹 Processing slide: TCGA-BA-5151-01Z-00-DX1
✅ Saved predicted expression: (1444, 12009)

🔹 Processing slide: TCGA-BA-5153-01Z-00-DX1
✅ Saved predicted expression: (2777, 12009)

🔹 Processing slide: TCGA-BA-6870-01Z-00-DX1
✅ Saved predicted expression: (157, 12009)

🔹 Processing slide: TCGA-BA-7269-01Z-00-DX1
✅ Saved predicted expression: (6126, 12009)

🔹 Processing slide: TCGA-BA-7269-01Z-00-DX1.npy.tmp
✅ Saved predicted expression: (6126, 12009)

🔹 Processing slide: TCGA-BA-A6D8-01Z-00-DX1
✅ Saved predicted expression: (11278, 12009)

🔹 Processing slide: TCGA-BA-A6D8-01Z-00-DX1.npy.tmp
✅ Saved predicted expression

In [ ]:
## 슬라이드 레벨 예측 (Loki 방식)

Cell 11에서 만든 tile-level prediction 결과를 slide별로 평균내어 
slide-level prediction을 생성합니다.


In [10]:
import numpy as np
from pathlib import Path
from tqdm import tqdm

# paths
tile_pred_dir = Path("/project_antwerp/hbae/script/predicted_expression_value/Patch_level_Prediction/tcga_predicted_expression_loki")  # Cell 11에서 만든 tile-level prediction 결과
out_dir = Path("tcga_pred_expr_loki")                   # Loki 방식 슬라이드 레벨 예측 결과
out_dir.mkdir(exist_ok=True)

# Cell 11에서 만든 tile-level prediction 결과를 slide별로 평균내기
if not tile_pred_dir.exists():
    raise FileNotFoundError(
        f"❌ 폴더를 찾을 수 없습니다: {tile_pred_dir}\n"
        f"Cell 11을 먼저 실행하여 tile-level prediction을 생성해주세요."
    )

tile_pred_files = sorted(tile_pred_dir.glob("*.npy"))
# .tmp 파일 제외
tile_pred_files = [f for f in tile_pred_files if not f.name.endswith(".tmp")]

print(f"Tile-level prediction files: {len(tile_pred_files)}")

if len(tile_pred_files) == 0:
    print(f"⚠️ 경고: {tile_pred_dir} 폴더에 .npy 파일이 없습니다.")
    print(f"Cell 11을 먼저 실행하여 tile-level prediction을 생성해주세요.")
else:
    for tile_pred_file in tqdm(tile_pred_files):
        slide_id = tile_pred_file.stem
        out_path = out_dir / f"{slide_id}.npy"
        
        if out_path.exists():
            continue

        # 🔑 tile-level prediction 결과 로드 (num_tiles, G)
        tile_pred = np.load(tile_pred_file).astype(np.float32)
        
        # 🔑 slide-level prediction: 모든 tile의 평균
        slide_pred = tile_pred.mean(axis=0)  # (G,)
        
        np.save(out_path, slide_pred)

    print(f"\n✅ Done: TCGA slide-level predicted expression (Loki 방식) saved to {out_dir}")

Tile-level prediction files: 333


100%|██████████| 333/333 [00:00<00:00, 21260.42it/s]


✅ Done: TCGA slide-level predicted expression (Loki 방식) saved to tcga_pred_expr_loki


In [ ]:
## 슬라이드 레벨 예측 (Loki 방식)

슬라이드의 모든 패치를 평균내어 하나의 슬라이드 임베딩을 만들고, 
이를 사용하여 슬라이드 레벨 유전자 발현을 예측합니다.


In [7]:
import numpy as np
from pathlib import Path
from tqdm import tqdm

# paths
emb_dir = Path("tcga_embeddings")          # slide patch embeddings
out_dir = Path("tcga_pred_expr_loki")      # Loki 방식 슬라이드 레벨 예측 결과
out_dir.mkdir(exist_ok=True)

# 이미 로드된 데이터 사용 (Cell 15에서 로드한 것)
# train_text_emb: (N_train, 768)
# train_expr    : (N_train, G)

def l2norm(x, axis=-1, eps=1e-8):
    return x / (np.linalg.norm(x, axis=axis, keepdims=True) + eps)

# Loki 방식으로 슬라이드 레벨 예측
slide_files = sorted(emb_dir.glob("*.npy"))
print(f"Slides: {len(slide_files)}")

for p in tqdm(slide_files):
    slide_id = p.stem
    out_path = out_dir / f"{slide_id}.npy"
    if out_path.exists():
        continue

    # 패치 임베딩 로드 및 정규화
    patch_emb = np.load(p)                 # (num_patches, 768)
    patch_emb = l2norm(patch_emb)          # normalize patches

    # 🔑 슬라이드 임베딩: 모든 패치의 평균
    slide_emb = patch_emb.mean(axis=0)     # (768,)
    slide_emb = l2norm(slide_emb)          # normalize slide embedding

    # 🔑 Loki 방식: 모든 similarity 계산
    similarity = slide_emb @ train_text_emb.T  # (N_train,)
    similarity = similarity.reshape(1, -1)      # (1, N_train) - Loki 함수는 2D 배열 기대

    # 🔑 Loki PredEx 함수 사용
    pred = loki.predex.predict_st_gene_expr(similarity, train_expr)  # (1, G)
    pred = pred[0]  # (G,) - 1D로 변환

    np.save(out_path, pred)

print(f"\n✅ Done: TCGA slide-level predicted expression (Loki 방식) saved to {out_dir}")


Slides: 0


0it [00:00, ?it/s]


✅ Done: TCGA slide-level predicted expression (Loki 방식) saved to tcga_pred_expr_loki_average


In [12]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# =========================
# 설정
# =========================
PRED_DIR = Path("tcga_pred_expr_loki_average")  # Loki 방식 슬라이드 레벨 예측 결과
TCGA_RNA_CSV = "/project_antwerp/TCGA-HNSC/ref_file.csv"
PRED_GENE_LIST = "/project_antwerp/hbae/data/all_shared_genes.txt"
HEG_GENE_LIST = "/project_antwerp/hbae/data/high_expression_genes_all.txt"

def norm_sid(x):
    """슬라이드 ID 정규화"""
    return str(x).split(".")[0]

def norm_gene(g):
    """유전자 이름 정규화"""
    g = str(g).split(".")[0].strip().upper()
    return g

# =========================
# 1) RNA 데이터 로딩 및 정리
# =========================
print("="*60)
print("1. TCGA RNA 데이터 로딩")
print("="*60)

rna_df = pd.read_csv(TCGA_RNA_CSV)

# slide_id 정규화
if 'wsi_file_name' in rna_df.columns:
    rna_df['wsi_file_name'] = rna_df['wsi_file_name'].astype(str)
    rna_df['wsi_file_name'] = rna_df['wsi_file_name'].str.split('/').str[-1]
    rna_df['slide_id'] = rna_df['wsi_file_name'].map(norm_sid)
    rna_df = rna_df.set_index('slide_id')
elif 'slide_id' in rna_df.columns:
    rna_df['slide_id'] = rna_df['slide_id'].map(norm_sid)
    rna_df = rna_df.set_index('slide_id')
else:
    first_col = rna_df.columns[0]
    rna_df[first_col] = rna_df[first_col].map(norm_sid)
    rna_df = rna_df.set_index(first_col)

# 불필요한 컬럼 제거
if 'Unnamed: 0' in rna_df.columns:
    rna_df = rna_df.drop(columns=['Unnamed: 0'])
if 'patient_id' in rna_df.columns:
    rna_df = rna_df.drop(columns=['patient_id'])

# 숫자 컬럼만 선택
rna_expr_df = rna_df.select_dtypes(include=[np.number]).copy()

# RNA gene 이름: rna_ prefix 제거 및 정규화
rna_gene_names = np.array([c.replace("rna_", "") for c in rna_expr_df.columns], dtype=object)
rna_gene_norm = np.array([norm_gene(g) for g in rna_gene_names], dtype=object)

# RNA gene map (중복 있으면 첫번째만)
rna_gene_map = {}
for i, g in enumerate(rna_gene_norm):
    if g not in rna_gene_map:
        rna_gene_map[g] = i

# RNA slide map
rna_slide_map = {sid: i for i, sid in enumerate(rna_df.index)}

print(f"RNA slides: {len(rna_slide_map)}, RNA genes: {rna_expr_df.shape[1]}")

# =========================
# 2) PredEx gene list 로딩
# =========================
print("\n" + "="*60)
print("2. PredEx gene list 로딩")
print("="*60)

pred_genes_raw = [line.strip() for line in open(PRED_GENE_LIST) if line.strip()]
pred_gene_norm = np.array([norm_gene(g) for g in pred_genes_raw], dtype=object)

pred_gene_map = {}
for i, g in enumerate(pred_gene_norm):
    if g not in pred_gene_map:
        pred_gene_map[g] = i

print(f"PredEx genes: {len(pred_gene_norm)}")

# =========================
# 3) HEG gene list 로딩
# =========================
print("\n" + "="*60)
print("3. HEG gene list 로딩")
print("="*60)

with open(HEG_GENE_LIST, "r") as f:
    heg_genes = [line.strip() for line in f if line.strip()]
heg_genes = [norm_gene(g) for g in heg_genes]

print(f"HEG genes: {len(heg_genes)}")

# =========================
# 4) 공통 gene 및 slide 교집합
# =========================
print("\n" + "="*60)
print("4. 공통 gene 및 slide 매칭")
print("="*60)

common_genes = sorted(
    set(heg_genes)
    & set(rna_gene_map.keys())
    & set(pred_gene_map.keys())
)

print(f"✅ Common genes: {len(common_genes)}")
print(f"Example: {common_genes[:10]}")

rna_idx = np.array([rna_gene_map[g] for g in common_genes], dtype=np.int64)
pred_idx = np.array([pred_gene_map[g] for g in common_genes], dtype=np.int64)

# 공통 slide 교집합
pred_files = sorted(PRED_DIR.glob("*.npy"))
pred_sids = [p.stem.split(".")[0] for p in pred_files]
pred_sid_set = set(pred_sids)

common_sids = sorted(pred_sid_set & set(rna_slide_map.keys()))
print(f"\nPred slides: {len(pred_sid_set)}")
print(f"Common slides: {len(common_sids)}")

if len(common_sids) == 0:
    raise RuntimeError("공통 slide가 0입니다. 경로나 파일 이름을 확인하세요.")

# =========================
# 5) Y_true, Y_pred 만들기
# =========================
print("\n" + "="*60)
print("5. 평가 행렬 생성")
print("="*60)

N = len(common_sids)
G = len(common_genes)

Y_true = np.zeros((N, G), dtype=np.float32)
Y_pred = np.zeros((N, G), dtype=np.float32)

pred_path_map = {p.stem.split(".")[0]: p for p in pred_files}

for j, sid in enumerate(tqdm(common_sids, desc="Building matrices")):
    # RNA
    ridx = rna_slide_map[sid]
    rna_row = rna_expr_df.iloc[ridx].values.astype(np.float32)
    Y_true[j] = rna_row[rna_idx]

    # Pred (Loki 방식)
    pred_vec = np.load(pred_path_map[sid]).astype(np.float32)
    if pred_vec.ndim != 1:
        raise ValueError(f"{sid}: pred 벡터가 1D가 아님: {pred_vec.shape}")
    Y_pred[j] = pred_vec[pred_idx]

print(f"Y_true: {Y_true.shape}, Y_pred: {Y_pred.shape}")

# =========================
# 6) Overall PCC (flatten)
# =========================
print("\n" + "="*60)
print("6. 평가 메트릭 계산")
print("="*60)

x = Y_true.reshape(-1).astype(np.float64)
y = Y_pred.reshape(-1).astype(np.float64)

x = x - x.mean()
y = y - y.mean()
overall_pcc = (x @ y) / (np.sqrt((x @ x) * (y @ y)) + 1e-12)
print(f"📌 Overall PCC (flatten): {overall_pcc:.4f}")

# =========================
# 7) Gene-wise PCC (벡터화)
# =========================
Xt = Y_true.astype(np.float64)
Yp = Y_pred.astype(np.float64)

Xt_mean = Xt.mean(axis=0, keepdims=True)
Yp_mean = Yp.mean(axis=0, keepdims=True)

Xt_c = Xt - Xt_mean
Yp_c = Yp - Yp_mean

cov = (Xt_c * Yp_c).sum(axis=0)
stdx = np.sqrt((Xt_c * Xt_c).sum(axis=0))
stdy = np.sqrt((Yp_c * Yp_c).sum(axis=0))

gene_pcc = cov / (stdx * stdy + 1e-12)
gene_pcc[np.isnan(gene_pcc)] = 0.0

gene_mae = np.mean(np.abs(Y_true - Y_pred), axis=0)

print(f"📌 Gene-wise PCC mean: {gene_pcc.mean():.4f}")
print(f"📌 Gene-wise MAE mean: {gene_mae.mean():.4f}")

# top/bottom genes
topk = 20
top_idx = np.argsort(gene_pcc)[-topk:][::-1]
bot_idx = np.argsort(gene_pcc)[:topk]

print("\n✅ Top PCC genes (Loki 방식):")
for i in top_idx[:10]:
    print(f"  {common_genes[i]}: PCC={gene_pcc[i]:.4f}, MAE={gene_mae[i]:.4f}")

print("\n❌ Bottom PCC genes (Loki 방식):")
for i in bot_idx[:10]:
    print(f"  {common_genes[i]}: PCC={gene_pcc[i]:.4f}, MAE={gene_mae[i]:.4f}")

# =========================
# 8) 결과 저장
# =========================
print("\n" + "="*60)
print("7. 결과 저장")
print("="*60)

np.save("eval_loki_common_sids.npy", np.array(common_sids, dtype=object))
np.save("eval_loki_common_genes.npy", np.array(common_genes, dtype=object))
np.save("eval_loki_gene_pcc.npy", gene_pcc.astype(np.float32))
np.save("eval_loki_gene_mae.npy", gene_mae.astype(np.float32))

print("✅ 저장 완료:")
print("  - eval_loki_common_sids.npy")
print("  - eval_loki_common_genes.npy")
print("  - eval_loki_gene_pcc.npy")
print("  - eval_loki_gene_mae.npy")

print("\n" + "="*60)
print("✅ Loki 방식 평가 완료!")
print("="*60)


1. TCGA RNA 데이터 로딩
RNA slides: 461, RNA genes: 24778

2. PredEx gene list 로딩
PredEx genes: 12009

3. HEG gene list 로딩
HEG genes: 7481

4. 공통 gene 및 slide 매칭
✅ Common genes: 6105
Example: ['A2M', 'A2ML1', 'A4GALT', 'AACS', 'AADACL2', 'AASS', 'AATK', 'ABAT', 'ABCA1', 'ABCA12']

Pred slides: 331
Common slides: 331

5. 평가 행렬 생성


Building matrices: 100%|██████████| 331/331 [00:00<00:00, 1295.77it/s]


Y_true: (331, 6105), Y_pred: (331, 6105)

6. 평가 메트릭 계산
📌 Overall PCC (flatten): 0.6820
📌 Gene-wise PCC mean: 0.0087
📌 Gene-wise MAE mean: 1.4891

✅ Top PCC genes (Loki 방식):
  HTR7: PCC=0.2439, MAE=1.2388
  ELK3: PCC=0.2404, MAE=2.6457
  SLC29A2: PCC=0.2273, MAE=1.4074
  WDR62: PCC=0.2259, MAE=1.0793
  MLLT6: PCC=0.2247, MAE=1.7959
  TNNT1: PCC=0.2242, MAE=2.8741
  PIAS4: PCC=0.2205, MAE=2.1766
  GJA1: PCC=0.2203, MAE=3.8462
  PRRG1: PCC=0.2200, MAE=0.9262
  DHRS7: PCC=0.2166, MAE=2.5071

❌ Bottom PCC genes (Loki 방식):
  ANXA5: PCC=-0.2716, MAE=3.7854
  TSSK6: PCC=-0.2445, MAE=0.6534
  PLCB3: PCC=-0.2437, MAE=2.9967
  MMP10: PCC=-0.2401, MAE=2.9608
  SIRPA: PCC=-0.2395, MAE=2.7196
  ITGA3: PCC=-0.2355, MAE=3.0440
  SERPINE1: PCC=-0.2304, MAE=3.5151
  CYP26B1: PCC=-0.2236, MAE=1.5360
  MMP1: PCC=-0.2180, MAE=3.6101
  AJAP1: PCC=-0.2176, MAE=0.5246

7. 결과 저장
✅ 저장 완료:
  - eval_loki_common_sids.npy
  - eval_loki_common_genes.npy
  - eval_loki_gene_pcc.npy
  - eval_loki_gene_mae.npy

✅ Loki 방

In [ ]:
# =========================
# 결과 저장 (추가된 부분 ⭐)
# =========================

# 현재 실행한 method에 맞게 이름만 바꿔서 저장하세요
# 예: baseline / topk_relu / linear

METHOD_NAME = "baseline"  
# METHOD_NAME = "topk_relu"
# METHOD_NAME = "linear"

np.save("Y_true.npy", Y_true.astype(np.float32))
np.save(f"Y_pred_{METHOD_NAME}.npy", Y_pred.astype(np.float32))

# 이미 gene / slide 정보는 저장돼 있으므로 그대로 사용
np.save("gene_names.npy", np.array(common_genes, dtype=object))
np.save("eval_common_sids.npy", np.array(common_sids, dtype=object))

print("\n✅ Saved files:")
print(f" - Y_true.npy: {Y_true.shape}")
print(f" - Y_pred_{METHOD_NAME}.npy: {Y_pred.shape}")
print(f" - gene_names.npy: {len(common_genes)} genes")
print(f" - eval_common_sids.npy: {len(common_sids)} slides")


In [15]:
import numpy as np
from pathlib import Path

old_dir = Path("/project_antwerp/hbae/script/predicted_expression_value/Patch_level_Prediction/tcga_predicted_expression_loki")            # 예전 슬라이드 예측 폴더 (있으면)
new_dir = Path("tcga_pred_expr_loki_average")    # 지금 폴더

sid = sorted(new_dir.glob("*.npy"))[0].stem
old = np.load(old_dir / f"{sid}.npy")
new = np.load(new_dir / f"{sid}.npy")

print("sid:", sid)
print("max abs diff:", np.max(np.abs(old-new)))
print("mean abs diff:", np.mean(np.abs(old-new)))
print("corr:", np.corrcoef(old, new)[0,1])


sid: TCGA-4P-AA8J-01Z-00-DX1
max abs diff: 0.012447357
mean abs diff: 5.153518e-05
corr: 0.9999999921916227


In [16]:
import numpy as np
from pathlib import Path

tile_dir = Path("/project_antwerp/hbae/script/predicted_expression_value/Patch_level_Prediction/tcga_predicted_expression_loki")
sid = "TCGA-4P-AA8J-01Z-00-DX1"
tile = np.load(tile_dir / f"{sid}.npy")  # (P,G)

# 타일 간 변동성
print("P,G =", tile.shape)
print("mean std across tiles (avg over genes):", tile.std(axis=0).mean())
print("mean abs deviation from slide mean:", np.mean(np.abs(tile - tile.mean(axis=0))))


P,G = (11834, 12009)
mean std across tiles (avg over genes): 6.775622e-05
mean abs deviation from slide mean: 5.153518e-05


In [5]:
import numpy as np
from pathlib import Path

# =========================
# Config
# =========================
EMB_DIR = Path("tcga_embeddings")
OUT_DIR = Path("tcga_predicted_expression")
OUT_DIR.mkdir(exist_ok=True)

TOPK = 128           # 64~256 추천 (클수록 정확↑, 메모리/시간↑)
PATCH_CHUNK = 256    # 128~1024 (RAM/속도에 맞춰)
TRAIN_CHUNK = 20000  # 10000~50000 (RAM에 맞춰)
EPS = 1e-8

# =========================
# Load train-side data
# =========================
train_text_emb = np.load("train_text_embedding.npy").astype(np.float32)  # (Ntrain, 768)
train_expr     = np.load("/project_antwerp/hbae/data/combined_expression_matrix.npy").astype(np.float32)  # (Ntrain, G)

print("Train text:", train_text_emb.shape)
print("Train expr:", train_expr.shape)

# L2 normalize (cosine = dot product)
train_text_emb /= (np.linalg.norm(train_text_emb, axis=1, keepdims=True) + EPS)

Ntrain, D = train_text_emb.shape
G = train_expr.shape[1]

def predict_for_patch_chunk(img_chunk: np.ndarray) -> np.ndarray:
    """
    img_chunk: (P, 768) float32, L2-normalized
    return: (P, G)
    """
    P = img_chunk.shape[0]

    top_scores = np.full((P, TOPK), -np.inf, dtype=np.float32)
    top_indices = np.full((P, TOPK), -1, dtype=np.int32)

    # train chunk-wise similarity
    for t0 in range(0, Ntrain, TRAIN_CHUNK):
        t1 = min(t0 + TRAIN_CHUNK, Ntrain)
        tr = train_text_emb[t0:t1]               # (Tc, 768)
        S = img_chunk @ tr.T                     # (P, Tc)

        k = min(TOPK, S.shape[1])
        idx_part = np.argpartition(S, -k, axis=1)[:, -k:]          # (P, k)
        sc_part  = np.take_along_axis(S, idx_part, axis=1)         # (P, k)
        idx_part = idx_part + t0                                   # global idx

        cand_scores = np.concatenate([top_scores, sc_part], axis=1)
        cand_idx    = np.concatenate([top_indices, idx_part], axis=1)

        sel = np.argpartition(cand_scores, -TOPK, axis=1)[:, -TOPK:]
        top_scores  = np.take_along_axis(cand_scores, sel, axis=1)
        top_indices = np.take_along_axis(cand_idx, sel, axis=1)

    # ✅ 안정적인 weight: ReLU(sim) 후 정규화 (음수 cosine 때문에 sum이 0되는 문제 방지)
    w = np.maximum(top_scores, 0.0)
    w = w / (w.sum(axis=1, keepdims=True) + EPS)   # (P, TOPK)

    # gather expr: (P, TOPK, G) then weighted sum -> (P, G)
    gathered = train_expr[top_indices]             # (P, TOPK, G)
    pred = (w[:, :, None] * gathered).sum(axis=1).astype(np.float32)
    return pred

# =========================
# Slide loop
# =========================
for emb_file in sorted(EMB_DIR.glob("*.npy")):
    if emb_file.name.endswith(".npy.tmp"):
        continue

    slide_id = emb_file.stem
    out_path = OUT_DIR / f"{slide_id}.npy"

    # resume
    if out_path.exists():
        print(f"⏭️ Skip (exists): {slide_id}")
        continue

    print(f"\n🔹 Processing slide: {slide_id}")

    img_emb = np.load(emb_file).astype(np.float32)  # (num_patches, 768)
    img_emb /= (np.linalg.norm(img_emb, axis=1, keepdims=True) + EPS)

    P = img_emb.shape[0]
    pred_chunks = []

    for p0 in range(0, P, PATCH_CHUNK):
        p1 = min(p0 + PATCH_CHUNK, P)
        pred_chunks.append(predict_for_patch_chunk(img_emb[p0:p1]))

    pred_expr = np.vstack(pred_chunks)
    out_path = OUT_DIR / f"{slide_id}.npy"
    tmp_path = OUT_DIR / f"{slide_id}.tmp.npy"

    np.save(tmp_path, pred_expr)
    tmp_path.replace(out_path)

    print(f"✅ Saved predicted expression: {pred_expr.shape}")


print("✅ Done.")


Train text: (154763, 768)
Train expr: (154763, 12009)

🔹 Processing slide: TCGA-4P-AA8J-01Z-00-DX1
✅ Saved predicted expression: (11834, 12009)

🔹 Processing slide: TCGA-BA-4074-01Z-00-DX1
✅ Saved predicted expression: (1033, 12009)

🔹 Processing slide: TCGA-BA-4077-01Z-00-DX1
✅ Saved predicted expression: (1749, 12009)

🔹 Processing slide: TCGA-BA-5151-01Z-00-DX1
✅ Saved predicted expression: (1444, 12009)

🔹 Processing slide: TCGA-BA-5153-01Z-00-DX1
✅ Saved predicted expression: (2777, 12009)

🔹 Processing slide: TCGA-BA-6870-01Z-00-DX1
✅ Saved predicted expression: (157, 12009)

🔹 Processing slide: TCGA-BA-7269-01Z-00-DX1
✅ Saved predicted expression: (6126, 12009)

🔹 Processing slide: TCGA-BA-7269-01Z-00-DX1.npy.tmp
✅ Saved predicted expression: (6126, 12009)

🔹 Processing slide: TCGA-BA-A6D8-01Z-00-DX1
✅ Saved predicted expression: (11278, 12009)
⏭️ Skip (exists): TCGA-BA-A6D8-01Z-00-DX1.npy.tmp

🔹 Processing slide: TCGA-BA-A6DA-01Z-00-DX1
✅ Saved predicted expression: (9139, 1200

In [6]:
import numpy as np
from pathlib import Path
from tqdm import tqdm

# paths
emb_dir = Path("tcga_embeddings")          # slide patch embeddings
out_dir = Path("tcga_pred_expr")          # predicted expression output
out_dir.mkdir(exist_ok=True)

# already loaded:
# train_text_emb: (N_train, 768)
# train_expr    : (N_train, G)

def l2norm(x, axis=-1, eps=1e-8):
    return x / (np.linalg.norm(x, axis=axis, keepdims=True) + eps)

def predex_slide(slide_patch_emb, train_text_emb, train_expr, topk=2048, temperature=1.0):
    # 1) pool patches -> slide embedding
    slide_emb = slide_patch_emb.mean(axis=0)  # (768,)
    slide_emb = l2norm(slide_emb)

    # 2) similarity
    sim = (slide_emb @ train_text_emb.T) / temperature  # (N_train,)

    # 3) top-k
    if topk is not None and topk < sim.shape[0]:
        idx = np.argpartition(sim, -topk)[-topk:]
        sel_sim = sim[idx]
        # stable softmax
        sel_sim = sel_sim - sel_sim.max()
        w = np.exp(sel_sim)
        w = w / (w.sum() + 1e-8)             # (topk,)
        pred = w @ train_expr[idx]           # (G,)
    else:
        sim = sim - sim.max()
        w = np.exp(sim)
        w = w / (w.sum() + 1e-8)
        pred = w @ train_expr

    return pred.astype(np.float32)

# run all slides
slide_files = sorted(emb_dir.glob("*.npy"))
print("Slides:", len(slide_files))

for p in tqdm(slide_files):
    slide_id = p.stem
    out_path = out_dir / f"{slide_id}.npy"
    if out_path.exists():
        continue

    patch_emb = np.load(p)                 # (num_patches, 768)
    patch_emb = l2norm(patch_emb)          # normalize patches (good practice)

    pred = predex_slide(patch_emb, train_text_emb, train_expr, topk=2048, temperature=1.0)
    np.save(out_path, pred)

print("✅ Done: TCGA slide-level predicted expression saved to", out_dir)


Slides: 333


100%|██████████| 333/333 [00:27<00:00, 12.26it/s]

✅ Done: TCGA slide-level predicted expression saved to tcga_pred_expr


In [20]:
list(pred_dict.keys())[:10]
rna_df.index[:10].tolist()


[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [21]:
import numpy as np
from pathlib import Path

p = next(Path("tcga_pred_expr").glob("*.npy"))
x = np.load(p)
print(p.name, x.shape)


TCGA-WA-A7GZ-01Z-00-DX2.npy (12009,)


In [23]:
from pathlib import Path

heg_path = Path("/project_antwerp/hbae/data/high_expression_genes_all.txt")

with open(heg_path, "r") as f:
    heg_genes = [line.strip() for line in f if line.strip()]

print("HEG genes:", len(heg_genes), "example:", heg_genes[:5])


HEG genes: 7481 example: ['A2M', 'A2ML1', 'A4GALT', 'AACS', 'AADACL2']


In [28]:
import numpy as np
import pandas as pd

# -----------------------
# 0) RNA slide id 정규화
# -----------------------
RNA_SLIDE_COL = "wsi_file_name"

def norm_sid(x):
    return str(x).split(".")[0]

rna_df = rna_df.copy()
rna_df["sid_norm"] = rna_df[RNA_SLIDE_COL].map(norm_sid)
rna_df = rna_df.drop_duplicates("sid_norm")

rna_slide_map = {sid: i for i, sid in enumerate(rna_df["sid_norm"].values)}

# -----------------------
# 1) RNA expression: 숫자 컬럼만
# -----------------------
rna_expr_df = rna_df.select_dtypes(include=[np.number])

print("RNA expr shape:", rna_expr_df.shape)

# 🔴 핵심: rna_ prefix 제거
rna_genes_raw = np.array(
    [c.replace("rna_", "") for c in rna_expr_df.columns],
    dtype=object
)

print("RNA genes example:", rna_genes_raw[:10])


RNA expr shape: (461, 24778)
RNA genes example: ['TSPAN6' 'DPM1' 'SCYL3' 'C1orf112' 'FGR' 'CFH' 'FUCA2' 'GCLC' 'NFYA'
 'STPG1']


In [29]:
# PredEx에서 사용한 gene list
pred_genes_raw = np.array(
    [g.strip() for g in open("/project_antwerp/hbae/data/all_shared_genes.txt") if g.strip()],
    dtype=object
)

print("Pred genes example:", pred_genes_raw[:10])
print("Pred genes count:", len(pred_genes_raw))


Pred genes example: ['A1CF' 'A2M' 'A2ML1' 'A3GALT2' 'A4GALT' 'A4GNT' 'AACS' 'AADAC' 'AADACL2'
 'AADACL3']
Pred genes count: 12009


In [33]:
def norm_gene(g):
    g = str(g)
    g = g.split(".")[0]      # 혹시 ENSG.xxx.15 같은 거 대비
    g = g.strip().upper()
    return g

rna_norm  = np.array([norm_gene(g) for g in rna_genes_raw], dtype=object)
pred_norm = np.array([norm_gene(g) for g in pred_genes_raw], dtype=object)

rna_gene_map = {}
for i, g in enumerate(rna_norm):
    if g not in rna_gene_map:
        rna_gene_map[g] = i

common_genes = sorted(
    set(heg_genes)
    & set(rna_gene_map.keys())
    & set(pred_gene_map.keys())
)

print("✅ Common HEG genes:", len(common_genes))


print("✅ Common genes:", len(common_genes))
print("Example:", common_genes[:10])


✅ Common HEG genes: 6095
✅ Common genes: 6095
Example: ['A2M', 'A2ML1', 'A4GALT', 'AACS', 'AADACL2', 'AASS', 'AATK', 'ABAT', 'ABCA1', 'ABCA12']


In [37]:
rna_idx  = np.array([rna_gene_map[g]  for g in common_genes], dtype=np.int64)
pred_idx = np.array([pred_gene_map[g] for g in common_genes], dtype=np.int64)


In [40]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# -----------------------
# 설정
# -----------------------
RNA_SLIDE_COL = "wsi_file_name"  # 너가 말한 컬럼명
PRED_DIR = Path("tcga_pred_expr")  # 네 pred slide-level .npy 저장한 폴더로 맞춰줘
PRED_GENE_LIST = "/project_antwerp/hbae/data/all_shared_genes.txt"

def norm_sid(x):
    return str(x).split(".")[0]

def norm_gene(g):
    g = str(g).split(".")[0].strip().upper()
    return g

# -----------------------
# 1) RNA: slide id + numeric expression만
# -----------------------
rna_df2 = rna_df.copy()
rna_df2["sid_norm"] = rna_df2[RNA_SLIDE_COL].map(norm_sid)
rna_df2 = rna_df2.drop_duplicates("sid_norm")

# 숫자 컬럼만 뽑기 (문자열 컬럼 TCGA-HNSC 같은거 제거됨)
rna_expr_df = rna_df2.select_dtypes(include=[np.number]).copy()

# RNA gene 이름: rna_ prefix 제거
rna_gene_names = np.array([c.replace("rna_", "") for c in rna_expr_df.columns], dtype=object)
rna_gene_norm  = np.array([norm_gene(g) for g in rna_gene_names], dtype=object)

# RNA gene map (중복 있으면 첫번째만)
rna_gene_map = {}
for i, g in enumerate(rna_gene_norm):
    if g not in rna_gene_map:
        rna_gene_map[g] = i

# RNA slide map
rna_slide_map = {sid: i for i, sid in enumerate(rna_df2["sid_norm"].values)}

print("RNA slides:", len(rna_slide_map), "RNA genes(numeric):", rna_expr_df.shape[1])

# -----------------------
# 2) Pred: gene list (12009) 로드
# -----------------------
pred_genes_raw = [line.strip() for line in open(PRED_GENE_LIST) if line.strip()]
pred_gene_norm = np.array([norm_gene(g) for g in pred_genes_raw], dtype=object)

pred_gene_map = {}
for i, g in enumerate(pred_gene_norm):
    if g not in pred_gene_map:
        pred_gene_map[g] = i

print("Pred genes:", len(pred_gene_norm))

# -----------------------
# 3) 공통 gene 만들기 (10331 나와야 정상)
# -----------------------
common_genes = sorted(
    set(heg_genes)
    & set(rna_gene_map.keys())
    & set(pred_gene_map.keys())
)

print("✅ Common genes:", len(common_genes), "Example:", common_genes[:10])

rna_idx  = np.array([rna_gene_map[g]  for g in common_genes], dtype=np.int64)
pred_idx = np.array([pred_gene_map[g] for g in common_genes], dtype=np.int64)

# -----------------------
# 4) 공통 slide 교집합 만들기
# -----------------------
pred_files = sorted(PRED_DIR.glob("*.npy"))
pred_sids = [p.stem.split(".")[0] for p in pred_files]  # 혹시 .npy.tmp 같은거 대비
pred_sid_set = set(pred_sids)

common_sids = sorted(pred_sid_set & set(rna_slide_map.keys()))
print("Pred slides:", len(pred_sid_set))
print("Common slides:", len(common_sids))

if len(common_sids) == 0:
    raise RuntimeError("공통 slide가 0이야. sid_norm 로직이나 pred 파일 이름 확인 필요")

# -----------------------
# 5) Y_true, Y_pred 만들기 (N_slides, N_common_genes)
# -----------------------
N = len(common_sids)
G = len(common_genes)

Y_true = np.zeros((N, G), dtype=np.float32)
Y_pred = np.zeros((N, G), dtype=np.float32)

# pred 파일 빠르게 찾기 dict
pred_path_map = {p.stem.split(".")[0]: p for p in pred_files}

for j, sid in enumerate(tqdm(common_sids, desc="Building matrices")):
    # RNA
    ridx = rna_slide_map[sid]
    rna_row = rna_expr_df.iloc[ridx].values.astype(np.float32)  # (RNA_numeric_genes,)
    Y_true[j] = rna_row[rna_idx]

    # Pred (12009,)
    pred_vec = np.load(pred_path_map[sid]).astype(np.float32)
    if pred_vec.ndim != 1:
        raise ValueError(f"{sid}: pred 벡터가 1D가 아님: {pred_vec.shape}")
    Y_pred[j] = pred_vec[pred_idx]

print("Y_true:", Y_true.shape, "Y_pred:", Y_pred.shape)

# -----------------------
# 6) Overall PCC (flatten)
# -----------------------
x = Y_true.reshape(-1).astype(np.float64)
y = Y_pred.reshape(-1).astype(np.float64)

x = x - x.mean()
y = y - y.mean()
overall_pcc = (x @ y) / (np.sqrt((x @ x) * (y @ y)) + 1e-12)
print(f"\n📌 Overall PCC (flatten): {overall_pcc:.4f}")

# -----------------------
# 7) Gene-wise PCC (벡터화)
# -----------------------
Xt = Y_true.astype(np.float64)
Yp = Y_pred.astype(np.float64)

Xt_mean = Xt.mean(axis=0, keepdims=True)
Yp_mean = Yp.mean(axis=0, keepdims=True)

Xt_c = Xt - Xt_mean
Yp_c = Yp - Yp_mean

cov = (Xt_c * Yp_c).sum(axis=0)
stdx = np.sqrt((Xt_c * Xt_c).sum(axis=0))
stdy = np.sqrt((Yp_c * Yp_c).sum(axis=0))

gene_pcc = cov / (stdx * stdy + 1e-12)
gene_pcc[np.isnan(gene_pcc)] = 0.0

gene_mae = np.mean(np.abs(Y_true - Y_pred), axis=0)

print(f"📌 Gene-wise PCC mean: {gene_pcc.mean():.4f}")
print(f"📌 Gene-wise MAE mean: {gene_mae.mean():.4f}")

# top/bottom genes
topk = 20
top_idx = np.argsort(gene_pcc)[-topk:][::-1]
bot_idx = np.argsort(gene_pcc)[:topk]

print("\n✅ Top PCC genes:")
for i in top_idx[:10]:
    print(f"  {common_genes[i]}: PCC={gene_pcc[i]:.4f}, MAE={gene_mae[i]:.4f}")

print("\n❌ Bottom PCC genes:")
for i in bot_idx[:10]:
    print(f"  {common_genes[i]}: PCC={gene_pcc[i]:.4f}, MAE={gene_mae[i]:.4f}")

# 필요하면 저장
np.save("eval_common_sids.npy", np.array(common_sids, dtype=object))
np.save("eval_common_genes.npy", np.array(common_genes, dtype=object))
np.save("eval_gene_pcc.npy", gene_pcc.astype(np.float32))
np.save("eval_gene_mae.npy", gene_mae.astype(np.float32))
print("\n✅ Saved: eval_common_sids.npy, eval_common_genes.npy, eval_gene_pcc.npy, eval_gene_mae.npy")


RNA slides: 461 RNA genes(numeric): 24778
Pred genes: 12009
✅ Common genes: 6095 Example: ['A2M', 'A2ML1', 'A4GALT', 'AACS', 'AADACL2', 'AASS', 'AATK', 'ABAT', 'ABCA1', 'ABCA12']
Pred slides: 331
Common slides: 331


Building matrices: 100%|██████████| 331/331 [00:00<00:00, 1432.52it/s]

Y_true: (331, 6095) Y_pred: (331, 6095)

📌 Overall PCC (flatten): 0.6879
📌 Gene-wise PCC mean: -0.0019
📌 Gene-wise MAE mean: 1.4451

✅ Top PCC genes:
  PTPN18: PCC=0.2494, MAE=1.8371
  REXO2: PCC=0.2440, MAE=1.7215
  DOK4: PCC=0.2335, MAE=2.0253
  NRBF2: PCC=0.2328, MAE=2.8679
  SLC25A19: PCC=0.2253, MAE=0.9597
  THSD1: PCC=0.2229, MAE=1.8790
  ELK3: PCC=0.2182, MAE=2.6340
  SLC29A2: PCC=0.2144, MAE=1.3768
  GZMH: PCC=0.2140, MAE=1.3225
  ULBP3: PCC=0.2134, MAE=1.5378

❌ Bottom PCC genes:
  PLEC: PCC=-0.2538, MAE=2.9696
  TSSK6: PCC=-0.2334, MAE=0.6460
  PTHLH: PCC=-0.2258, MAE=2.8899
  MGAT3: PCC=-0.2198, MAE=0.5054
  SIRPA: PCC=-0.2190, MAE=2.7116
  TTPAL: PCC=-0.2177, MAE=1.9380
  ATN1: PCC=-0.2125, MAE=2.9989
  FSTL3: PCC=-0.2056, MAE=2.5273
  SH2D5: PCC=-0.2019, MAE=1.2624
  ANXA5: PCC=-0.2005, MAE=3.6249

✅ Saved: eval_common_sids.npy, eval_common_genes.npy, eval_gene_pcc.npy, eval_gene_mae.npy


In [5]:
print("ref_file genes:", len(ref_genes))          # ref_file에서 읽은 gene 목록
print("rna_df genes:", len(rna_df.columns))       # TCGA expression matrix 컬럼 수
print("pred_df genes:", len(pred_df.columns))     # 네 예측 결과 컬럼 수
print("shared_genes:", len(shared_genes))         # 최종 비교 gene 수 (=12009)


NameError: name 'ref_genes' is not defined